# ReACT

Exercise 2: Generate Python code, execute tests, observe failures, and repair the code.


## Method

Tools used: Codex for AI-authored prompts, responses, and Python code; Python 3 for execution; Google Colab for the notebook interface; GitHub for distribution.

The AI text was authored in the Codex session that prepared this exercise. The notebook stores those responses so the example can be replayed without an API account. Running it does **not** make a new model call. The prompts below are the exercise instructions used to guide the AI-authored examples, not a transcript of the platform's internal system instructions. Python execution and printed test results are real.

Open the notebook in Colab and choose **Runtime > Run all**. No packages, credentials, or data downloads are needed. To try a different case, give the displayed prompts and new inputs to an AI, record the new responses, and rerun the checks. Editing the input alone is not a fresh AI experiment.


## Prompt

Exercise role instruction:

You are a Python coding assistant. Use brief implementation plans and observable test results. Separate PLAN, ACTION, OBSERVATION, and FIX. Do not claim execution until Python has returned a result. Use only the Python standard library.

Full user prompt:

```text
Build mean_score(scores) for a nonempty list or tuple of finite int or float values.
Return a float arithmetic mean. Reject booleans, nonnumeric elements, and non-list/tuple inputs with TypeError.
Reject empty inputs and nonfinite values with ValueError. Do not mutate the input. Use only the standard library.
Use this ReACT cycle:
PLAN: give a brief implementation plan and list edge cases.
ACTION: generate Python code. For the first pass, implement only the basic arithmetic path so the test feedback can guide validation.
RUN: execute the supplied tests, including invalid inputs. Catch expected prototype failures in the test report.
OBSERVATION: report actual pass/fail results without inventing output.
FIX: use the observed failures to revise the code to meet the full contract.
RUN AGAIN: execute the same tests and print the final sample result and passing count.
Keep code separate from the short explanation. Do not install packages, read files, or use the network.
```


## Plan

AI-authored implementation plan:

First implement sum divided by count. Test normal values, empty inputs, wrong types, booleans, and nonfinite values. Then use the observed failures to add explicit validation.

The first pass is deliberately minimal, as requested in the prompt. It is a teaching prototype; its test failures are retained as part of the iteration.


In [1]:
def mean_score(scores):
    return sum(scores) / len(scores)

mean_score_v1 = mean_score
print("ACTION: generated prototype")
print("Sample:", mean_score_v1([70, 80, 90]))


ACTION: generated prototype
Sample: 80.0


## Run

These tests check behavior independently of the implementation. Expected prototype failures are printed, so the notebook can continue to the repair. The exception types are part of the contract.


In [2]:
import math

CASES = [
    ('integers', [70, 80, 90], 80.0),
    ('floats', [1.5, 2.5], 2.0),
    ('tuple', (2, 4), 3.0),
    ('single zero', [0], 0.0),
    ('negatives', [-5, 5], 0.0),
    ('empty', [], ValueError),
    ('text element', [1, '2'], TypeError),
    ('boolean', [True, 2], TypeError),
    ('none input', None, TypeError),
    ('string input', '12', TypeError),
    ('nan', [1, float('nan')], ValueError),
    ('infinity', [1, float('inf')], ValueError),
    ('set input', {1, 3}, TypeError),
    ('large finite values', [1e308, 1e308], 1e308),
]

def run_tests(function):
    results = []
    for label, values, expected in CASES:
        before = values.copy() if isinstance(values, list) else values
        try:
            actual = function(values)
            if isinstance(expected, type) and issubclass(expected, Exception):
                passed = False
                detail = f'expected {expected.__name__}, got {actual!r}'
            else:
                passed = type(actual) is float and math.isclose(actual, expected)
                detail = f'expected {expected!r}, got {actual!r}'
        except Exception as error:
            passed = isinstance(expected, type) and type(error) is expected
            detail = f'got {type(error).__name__}: {error}'
        if isinstance(values, list) and values != before:
            passed, detail = False, 'input was mutated'
        results.append({'case': label, 'passed': passed, 'detail': detail})
        print(('PASS' if passed else 'FAIL') + f' | {label} | {detail}')
    print(f"{sum(row['passed'] for row in results)}/{len(results)} tests passed")
    return results

print("OBSERVATION: first pass")
initial_results = run_tests(mean_score_v1)


OBSERVATION: first pass
PASS | integers | expected 80.0, got 80.0
PASS | floats | expected 2.0, got 2.0
PASS | tuple | expected 3.0, got 3.0
PASS | single zero | expected 0.0, got 0.0
PASS | negatives | expected 0.0, got 0.0
FAIL | empty | got ZeroDivisionError: division by zero
PASS | text element | got TypeError: unsupported operand type(s) for +: 'int' and 'str'
FAIL | boolean | expected TypeError, got 1.5
PASS | none input | got TypeError: 'NoneType' object is not iterable
PASS | string input | got TypeError: unsupported operand type(s) for +: 'int' and 'str'
FAIL | nan | expected ValueError, got nan
FAIL | infinity | expected ValueError, got inf
FAIL | set input | expected TypeError, got 2.0
FAIL | large finite values | expected 1e+308, got inf
8/14 tests passed


## Fix

The repair prompt below incorporates the actual results returned by the previous cell. In the authoring session, Codex received this test feedback before writing the revision. During replay, no model is called.


In [3]:
fix_prompt = (
    'Revise mean_score to satisfy the full contract. Use explicit input validation and avoid '
    'overflow for [1e308, 1e308]. Preserve the function name and do not mutate inputs. '
    'Return a short fix note followed by Python code. Actual observations follow:\n'
)
for row in initial_results:
    fix_prompt += ('PASS' if row['passed'] else 'FAIL') + ' | ' + row['case'] + ' | ' + row['detail'] + '\n'
fix_prompt += f"{sum(row['passed'] for row in initial_results)}/{len(initial_results)} tests passed\n"
print(fix_prompt)


Revise mean_score to satisfy the full contract. Use explicit input validation and avoid overflow for [1e308, 1e308]. Preserve the function name and do not mutate inputs. Return a short fix note followed by Python code. Actual observations follow:
PASS | integers | expected 80.0, got 80.0
PASS | floats | expected 2.0, got 2.0
PASS | tuple | expected 3.0, got 3.0
PASS | single zero | expected 0.0, got 0.0
PASS | negatives | expected 0.0, got 0.0
FAIL | empty | got ZeroDivisionError: division by zero
PASS | text element | got TypeError: unsupported operand type(s) for +: 'int' and 'str'
FAIL | boolean | expected TypeError, got 1.5
PASS | none input | got TypeError: 'NoneType' object is not iterable
PASS | string input | got TypeError: unsupported operand type(s) for +: 'int' and 'str'
FAIL | nan | expected ValueError, got nan
FAIL | infinity | expected ValueError, got inf
FAIL | set input | expected TypeError, got 2.0
FAIL | large finite values | expected 1e+308, got inf
8/14 tests passed

AI-authored repair note:

The prototype passed 8/14 tests. Validate the container, reject empty inputs and booleans, check numeric finiteness, and divide each value by the count before summing to avoid the observed overflow. Rerun the same 14 tests.


In [4]:
def mean_score(scores):
    """Return the mean of a nonempty list/tuple of finite numeric scores."""
    if not isinstance(scores, (list, tuple)):
        raise TypeError('scores must be a list or tuple')
    if not scores:
        raise ValueError('scores must not be empty')
    for value in scores:
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError('each score must be an int or float, not bool')
        if isinstance(value, float) and not math.isfinite(value):
            raise ValueError('scores must be finite')
    try:
        result = math.fsum(value / len(scores) for value in scores)
    except OverflowError as error:
        raise ValueError('scores exceed the supported float range') from error
    if not math.isfinite(result):
        raise ValueError('mean exceeds the supported float range')
    return result

print("ACTION: revised function defined")


ACTION: revised function defined


## Rerun

The same cases run again. The final assertion makes an unresolved failure stop execution.


In [5]:
print('OBSERVATION: repaired version')
final_results = run_tests(mean_score)
assert all(row['passed'] for row in final_results), 'A final test failed'
assert sum(row['passed'] for row in initial_results) < len(initial_results)
print('FINAL OUTPUT:', mean_score([70, 80, 90]))
print('PASS: complete PLAN -> ACTION -> RUN -> OBSERVE -> FIX -> RUN cycle')


OBSERVATION: repaired version
PASS | integers | expected 80.0, got 80.0
PASS | floats | expected 2.0, got 2.0
PASS | tuple | expected 3.0, got 3.0
PASS | single zero | expected 0.0, got 0.0
PASS | negatives | expected 0.0, got 0.0
PASS | empty | got ValueError: scores must not be empty
PASS | text element | got TypeError: each score must be an int or float, not bool
PASS | boolean | got TypeError: each score must be an int or float, not bool
PASS | none input | got TypeError: scores must be a list or tuple
PASS | string input | got TypeError: scores must be a list or tuple
PASS | nan | got ValueError: scores must be finite
PASS | infinity | got ValueError: scores must be finite
PASS | set input | got TypeError: scores must be a list or tuple
PASS | large finite values | expected 1e+308, got 1e+308
14/14 tests passed
FINAL OUTPUT: 80.0
PASS: complete PLAN -> ACTION -> RUN -> OBSERVE -> FIX -> RUN cycle


## Review

The prototype used Python's default behavior, so an empty list raised the wrong exception and booleans were treated as numbers. The revision enforces the input contract. Dividing values before `math.fsum` fixes the tested large-value overflow. The final version passes the same 14 tests.

Limits: results use floating-point arithmetic, so small rounding differences are possible. Extremely large integers outside the supported float range raise ValueError. Passing these cases is evidence for this function, not proof for every numeric input.
